# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — point this at your data

*Adjust `DATA_DIR` to wherever your repo keeps the parquet files (e.g. after the `load_dataset`/DuckDB pull from the README).*

In [3]:
import pandas as pd
import numpy as np

DATA_DIR = "../../"

perf = pd.read_parquet(
    f"{DATA_DIR}/fact_content_daily_performance_sample.parquet",
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

dc = pd.read_parquet(
    f"{DATA_DIR}/dim_content.parquet",
    columns=[
        "client_hash_id",
        "content_hash_id",
        "content_updated_date",
        "search_volume",
        "is_published",
        "is_deleted"
    ]
)

perf["report_date"] = pd.to_datetime(perf["report_date"])
dc["content_updated_date"] = pd.to_datetime(dc["content_updated_date"])

print("perf:", perf.shape, perf["report_date"].min(), "to", perf["report_date"].max())
print("dim_content:", dc.shape)

perf: (11694072, 6) 2026-06-01 00:00:00 to 2026-06-30 00:00:00
dim_content: (519606, 6)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1 — CTR vs. position (flag-linked: FlyRank's CTR-fix logic).**
Verdict: **CONFIRMED**. CTR falls off a cliff after position 3 — the 1-3 bucket converts at
4.41%, roughly 6-13x every other bucket (3-5: 0.68%, 5-10: 0.35%, 10-20: 0.41%, 20-50: 0.26%,
50+: 0.06%). There's a small non-monotonic bump (10-20 sits a touch above 5-10), but with
30k-37k rows in each of those buckets that's noise around a flat middle, not a real reversal.
The dominant pattern — top-3 positions convert far better than anything else — holds and is
strong enough to build a benchmark curve on.

**Signal 2 — staleness vs. within-month click decline.**
Verdict: **FALSE**. `pct_declined` is flat across every staleness bucket: 54.0% (≤30d), 52.7%
(30-90d), 53.4% (90-180d), 52.4% (180-365d). If staleness drove decline I'd expect that
percentage to climb with days-since-update — it doesn't move at all. There's no usable signal
here, so I'm not building the rule around "stale content declines more" — the data doesn't
support it.

**Why the rule changed from my original plan.** I went in expecting a
`HIGH_IMP_LOW_CTR_STALE` rule. Signal 2 killed the staleness half, so the rule below drops
staleness entirely and scores on the one signal that actually held up: how far a page's CTR
sits below the typical CTR for its own position bucket.

**The rule, in plain English:**
- **Population scored:** content already ranking on page 1-2 (`avg_pos` ≤ 20) with at least
  50 impressions this month — high enough traffic to trust the CTR estimate, and a position
  range where a title/meta rewrite is a plausible fix rather than a ranking problem.
- **Score:** `(expected_ctr_for_position_bucket − actual_ctr) × impressions` — the CTR gap,
  weighted by how much traffic that gap is actually costing. `expected_ctr` is this month's own
  observed CTR-by-position-bucket curve (Signal 1's table) — no future window, no product flags.
- **Reason code (one, always):** `CTR_GAP_VS_POSITION`.
- **Action:** `REWRITE_TITLE_META` — the page already ranks; the snippet isn't earning the
  clicks its position should get.


In [4]:
# Signal check 1: CTR vs. position bucket table (behind FlyRank's CTR-fix logic)
agg = perf.groupby(["client_hash_id","content_hash_id"]).agg(
    impressions=("gsc_impressions","sum"),
    clicks=("gsc_clicks","sum"),
    avg_pos=("gsc_avg_position","mean"),
).reset_index()
agg = agg[agg.impressions >= 50].copy()   # drop near-zero-impression noise
agg["ctr"] = agg.clicks / agg.impressions

bins = [0, 3, 5, 10, 20, 50, 1000]
labels = ["1-3", "3-5", "5-10", "10-20", "20-50", "50+"]
agg["pos_bucket"] = pd.cut(agg.avg_pos, bins=bins, labels=labels).astype(str)

bucket_table = agg.groupby("pos_bucket").agg(
    n=("content_hash_id", "count"),
    total_impressions=("impressions", "sum"),
    total_clicks=("clicks", "sum"),
)
bucket_table["ctr"] = bucket_table.total_clicks / bucket_table.total_impressions
bucket_table = bucket_table.loc[labels]  # keep natural position order
print(bucket_table)

                n  total_impressions  total_clicks       ctr
pos_bucket                                                  
1-3          2199            8047356        355059  0.044121
3-5          8894           36242891        246128  0.006791
5-10        37261          116740405        412144  0.003530
10-20       30258           29838629        122402  0.004102
20-50       31204           21347187         56194  0.002632
50+         10863            2824095          1820  0.000644


In [5]:
# Signal check 2: staleness vs. within-month decline (fill in after you write your verdict above)
half = np.where(perf.report_date <= perf.report_date.min() + pd.Timedelta(days=14), "h1", "h2")
perf2 = perf.assign(half=half)
trend = perf2.groupby(["client_hash_id","content_hash_id","half"]).agg(clicks=("gsc_clicks","sum")).reset_index()
piv = trend.pivot_table(index=["client_hash_id","content_hash_id"], columns="half", values="clicks", fill_value=0).reset_index()
piv = piv[(piv.h1 > 0) | (piv.h2 > 0)]

m = piv.merge(dc[dc.is_published & ~dc.is_deleted][["client_hash_id","content_hash_id","content_updated_date"]],
              on=["client_hash_id","content_hash_id"], how="inner").dropna(subset=["content_updated_date"])
month_end = perf.report_date.max()
m["days_since_update"] = (month_end - m.content_updated_date).dt.days

bins = [-1, 30, 90, 180, 365, 3650]
labels = ["<=30d", "30-90d", "90-180d", "180-365d", "365d+"]
m["staleness_bucket"] = pd.cut(m.days_since_update, bins=bins, labels=labels)
m["clicks_declined"] = m.h2 < m.h1

staleness_table = m.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_h1_clicks=("h1", "mean"),
    avg_h2_clicks=("h2", "mean"),
    pct_declined=("clicks_declined", "mean"),
)
print(staleness_table)

                      n  avg_h1_clicks  avg_h2_clicks  pct_declined
staleness_bucket                                                   
<=30d             26736       7.893103       6.574955      0.539984
30-90d            25884       4.754057       4.101955      0.526773
90-180d            8293       3.261184       2.499457      0.533583
180-365d             84       2.476190       2.071429      0.523810


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule implemented below: **CTR gap vs. an expected-CTR-by-position benchmark, weighted by impressions.**
Only content already ranking on page 1-2 (position ≤ 20) with impressions ≥ 50 is scored — that's the population where a title/meta rewrite is plausibly the fix, not a ranking problem. Expected CTR is a benchmark built from this same month's data (no future window, no product flags used).

In [6]:
expected_ctr_by_bucket = bucket_table["ctr"]  # from section 1 — this month's own benchmark curve

agg["expected_ctr"] = agg.pos_bucket.map(expected_ctr_by_bucket).astype(float)
agg["ctr_gap"] = agg.expected_ctr - agg.ctr

dc_live = dc[dc.is_published & ~dc.is_deleted]
scored = agg.merge(dc_live[["client_hash_id","content_hash_id","search_volume"]],
                    on=["client_hash_id","content_hash_id"], how="inner")

scored = scored[(scored.avg_pos <= 20) & (scored.ctr_gap > 0)].copy()
scored["action_score"] = scored.ctr_gap * scored.impressions
scored["reason_code"] = "CTR_GAP_VS_POSITION"
scored["action"] = "REWRITE_TITLE_META"

scored = scored.sort_values("action_score", ascending=False).reset_index(drop=True)
print("rows scored:", len(scored))
scored.head(10)

rows scored: 45854


,client_hash_id,content_hash_id,impressions,clicks,avg_pos,ctr,pos_bucket,expected_ctr,ctr_gap,search_volume,action_score,reason_code,action
0,client_e547b89c05043229,content_545bb6cc7081ded3,585712,3048,2.110776,0.005204,1-3,0.044121,0.038917,40.0,22794.316036,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
1,client_e547b89c05043229,content_eadb33b5df496f4a,591696,3817,2.258612,0.006451,1-3,0.044121,0.037670,390.0,22289.337294,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
2,client_8ddc46da5414ffd8,content_943dc881428182b8,292416,399,2.689206,0.001364,1-3,0.044121,0.042757,20.0,12502.744690,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
3,client_e547b89c05043229,content_9ef3d7516483e665,269943,787,2.098715,0.002915,1-3,0.044121,0.041206,70.0,11123.208973,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
4,client_8ddc46da5414ffd8,content_cca099da6c658785,157289,631,1.770567,0.004012,1-3,0.044121,0.040109,880.0,6308.779357,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
5,client_e547b89c05043229,content_0e03de7680314cd5,119228,359,2.388663,0.003011,1-3,0.044121,0.041110,110.0,4901.482381,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
6,client_e547b89c05043229,content_61215c724c8220ae,139884,1402,1.802033,0.010023,1-3,0.044121,0.034099,10.0,4769.849879,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
7,client_73cda7b4e4f265ea,content_e9856d7d976aa034,116908,550,1.860472,0.004705,1-3,0.044121,0.039417,20.0,4608.121198,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
8,client_e547b89c05043229,content_8d7d99f109e19aa2,103230,445,2.177460,0.004311,1-3,0.044121,0.039810,70.0,4109.631430,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
9,client_8ddc46da5414ffd8,content_7471467133493ce6,75720,53,2.793545,0.000700,1-3,0.044121,0.043421,4400.0,3287.857231,CTR_GAP_VS_POSITION,REWRITE_TITLE_META


In [7]:
import os
os.makedirs("work/outputs", exist_ok=True)
scored.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv —", scored.shape)

wrote work/outputs/baseline_action_score.csv — (45854, 13)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

All 20 rows carry the same action (`REWRITE_TITLE_META`) and reason code
(`CTR_GAP_VS_POSITION`) — that's expected, since it's one rule. What varies is the underlying
story, and that's where I'd push back on the score rather than trust it blindly.

**What's actually driving the top of the queue:** these are pages already sitting at position
~1.8–2.9 (essentially the top of page 1) with impressions from 585k down to ~52k, all
converting far below the 4.4% a position-1-3 page "should" get. The action score rewards raw
impressions, so the queue is dominated by high-traffic pages even when their CTR gap
(percentage-point-wise) isn't the largest in the set — e.g. row 8 (`content_8d7d99f109e19aa2`)
has a smaller gap than row 11 (`content_2752442d8c27de3b`) but outranks it purely on volume.
That's a deliberate design choice (fix the biggest traffic leak first), not a bug, but it's
worth being explicit that this is an *impact* ranking, not a *severity* ranking.

**Rows I'd sanity-check before acting on, specifically:**
- **Row 9 (`content_7471467133493ce6`, search_volume 4,400 — the highest in the top 20)**:
  CTR is 0.07% despite ~2.8 average position and 75k impressions. That combination (good
  position, real query volume, near-zero clicks) is the classic signature of a SERP feature
  (featured snippet, People Also Ask, a knowledge panel) eating the click before the user ever
  scrolls to organic — or a branded/navigational query where the searcher already knows where
  they're going. A rewrite won't fix either of those. I'd want SERP-feature presence as a flag
  before trusting this one.
- **Row 15 (`content_9a4459a8a3b7a514`)**: CTR here is 1.63% — the best-performing page in the
  whole top 20, by a wide margin. It only lands in the queue because 4.41% is a high bar (it's
  pulled up by the position-1-3 bucket's biggest outliers, per the self-referential-benchmark
  note in cell 8). This page isn't obviously broken; the gap is more "below an ambitious
  average" than "underperforming."
- **Rows with `search_volume = 0`** (`content_145680ddd5f91ea9`, `content_3b8638bf882420e3`,
  `content_c556c7369fb2fd06`, `content_d648d5ef74c87d7d`): each still pulls 50k+ impressions.
  Zero recorded search volume next to real impression volume means these pages are earning
  traffic from a long tail of queries the `search_volume` field doesn't capture (or the field
  is stale/missing for them) — the context column I'm using for human review is unreliable
  here, even though it isn't part of the score itself.

**What would make any of these wrong, generally:** a SERP feature or PAA box absorbing clicks
at that position, a branded/navigational query where low CTR is normal and expected, a
seasonal or declining-demand query where `search_volume` is stale, or a redirect/canonical
situation where the impressions are really landing on a different URL than the one scored.


In [8]:
top20 = scored.head(20)[["client_hash_id","content_hash_id","avg_pos","ctr","expected_ctr",
                          "ctr_gap","impressions","search_volume","action_score","reason_code","action"]]
top20

,client_hash_id,content_hash_id,avg_pos,ctr,expected_ctr,ctr_gap,impressions,search_volume,action_score,reason_code,action
0,client_e547b89c05043229,content_545bb6cc7081ded3,2.110776,0.005204,0.044121,0.038917,585712,40.0,22794.316036,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
1,client_e547b89c05043229,content_eadb33b5df496f4a,2.258612,0.006451,0.044121,0.037670,591696,390.0,22289.337294,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
2,client_8ddc46da5414ffd8,content_943dc881428182b8,2.689206,0.001364,0.044121,0.042757,292416,20.0,12502.744690,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
3,client_e547b89c05043229,content_9ef3d7516483e665,2.098715,0.002915,0.044121,0.041206,269943,70.0,11123.208973,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
4,client_8ddc46da5414ffd8,content_cca099da6c658785,1.770567,0.004012,0.044121,0.040109,157289,880.0,6308.779357,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
5,client_e547b89c05043229,content_0e03de7680314cd5,2.388663,0.003011,0.044121,0.041110,119228,110.0,4901.482381,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
6,client_e547b89c05043229,content_61215c724c8220ae,1.802033,0.010023,0.044121,0.034099,139884,10.0,4769.849879,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
7,client_73cda7b4e4f265ea,content_e9856d7d976aa034,1.860472,0.004705,0.044121,0.039417,116908,20.0,4608.121198,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
8,client_e547b89c05043229,content_8d7d99f109e19aa2,2.177460,0.004311,0.044121,0.039810,103230,70.0,4109.631430,CTR_GAP_VS_POSITION,REWRITE_TITLE_META
9,client_8ddc46da5414ffd8,content_7471467133493ce6,2.793545,0.000700,0.044121,0.043421,75720,4400.0,3287.857231,CTR_GAP_VS_POSITION,REWRITE_TITLE_META


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks in the queue, and why:**
1. **Row 15 (`content_9a4459a8a3b7a514`)** — already covered above: 1.63% CTR is good in
   absolute terms. This page is being flagged mainly because the position-1-3 benchmark is
   pulled high by a handful of very-high-CTR outliers in that same bucket, not because this
   page is clearly failing. A model with a less impression-dominated benchmark (e.g. a median
   instead of a pooled mean) would likely rank this one lower or drop it.
2. **Row 9 (`content_7471467133493ce6`)** — flagged above for the SERP-feature / branded-query
   risk. This is the pick I'd least want an editor to act on without checking the SERP first,
   since a title/meta rewrite can't fix a snippet stealing the click.
3. **Client concentration** — 8 of the top 20 rows belong to a single client
   (`client_e547b89c05043229`), and 3 clients together account for 16 of 20. Because the score
   is impressions × gap, any client with high absolute traffic will structurally dominate the
   queue even if a smaller client has a proportionally worse CTR problem. As a *company-wide*
   action queue this is fine (fix the biggest leaks first); as a *per-client* editorial queue
   it would systematically starve smaller clients of attention. Worth naming as a known
   limitation rather than discovering it later.

**Leakage self-check:**
The scoring rule uses only `report_date`, `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`
from `fact_content_daily_performance_sample` (this month only — June 2026, the data's own
window, nothing forward-looking) and `search_volume`, `is_published`, `is_deleted` from
`dim_content` (context/filtering only, not part of the score). `content_updated_date` was used
only in the Signal 2 staleness check, and that signal was rejected (FALSE) and does not feed
the final score at all. `fact_content_query_90d` — the table with the documented label-period
overlap — is not touched anywhere in this notebook (confirmed in the cell above). No product
flags, no future-window data, and no label-derived fields are used. The one thing I'm not
fully certain about: whether `content_updated_date` itself could already reflect an automated
content-refresh action taken *because* of past performance (i.e., is it partly a consequence of
the thing I'm trying to predict, rather than a pure input?). I don't have visibility into how
that field gets set, so I'm flagging it as an open question rather than asserting it's clean —
it isn't used in scoring either way, so it doesn't affect this baseline, but it would matter if
a future model features it.


In [9]:
# Quick assist: columns actually used in scoring, for your leakage statement above
print("Columns used from fact_content_daily_performance_sample:", ["report_date","gsc_impressions","gsc_clicks","gsc_avg_position"])
print("Columns used from dim_content:", ["content_updated_date (signal check only)","search_volume","is_published","is_deleted"])
print("fact_content_query_90d used:", False)

Columns used from fact_content_daily_performance_sample: ['report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Columns used from dim_content: ['content_updated_date (signal check only)', 'search_volume', 'is_published', 'is_deleted']
fact_content_query_90d used: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.